In [ ]:
# Import libraries
import os
import csv
import gzip
import pickle
import numpy as np
import pandas as pd
import gseapy as gp
import seaborn as sns
import matplotlib.pyplot as plt
from datasets import load_dataset
from pxblat import Server, Client #pxblat==0.3.6

 # Load Data

In [ ]:
# Define model
model = '' # classification, pretrained or random_init
dataset = '' # ms or pancreas
approach = 'attention'

In [ ]:
# Load attention scores
base = '' # path to attention dir
data_dir = os.path.join(base, dataset, model)

pkl_files = [
    'examples_scores_attention_layer0.p',
    'examples_scores_attention_layer1.p',
    'examples_scores_attention_layer2.p',
    'examples_scores_attention_layer3.p',
    'examples_scores_attention_layer4.p',
    'examples_scores_attention_layer5.p',
    'examples_scores_attention_layer6.p',
    'examples_scores_attention_layer7.p',
    'examples_scores_attention_layer8.p',
    'examples_scores_attention_layer9.p',
    'examples_scores_attention_layer10.p',
    'examples_scores_attention_layer11.p',
]


In [ ]:
# Load all pickle files into memory
layers_data = []
for file_name in pkl_files:
    file_path = os.path.join(data_dir, file_name)
    with open(file_path, 'rb') as f:
        layers_data.append(pickle.load(f))

# Assuming all layers have the same number of heads and all heads have the same number of cells
num_layers = len(layers_data)
num_heads = len(layers_data[0])
num_cells = len(layers_data[0][0])
num_genes = len(layers_data[0][0][0][0]) - 1  # Assuming each cell contains data for the same number of genes

print('Check Data:', num_layers, num_heads, num_cells, num_genes)

# Attention Score Heatmap

In [ ]:
mean_score_df:pd.DataFrame = pd.DataFrame()

# Open the layer
for layer in range(12):
    print("LAYER:", layer)
    with open(f'{data_dir}/examples_scores_attention_layer{layer}.p', 'rb') as f:
        results:dict = pickle.load(f)

    # Calculate mean score per sequence per head
    tmp_dict:dict = {}
    for head in results:
        print("HEAD:", head)
        tmp_list:list = []
        for i in range(len(results[head])):
          tmp_list.append(np.mean((results[head][i][0])))
        tmp_dict[head] = tmp_list

    # Merge layer scores
    tmp_df:pd.DataFrame = pd.DataFrame(tmp_dict)
    tmp_df.columns = [f'head{i}' for i in range(8)]
    tmp_df['layer'] = f'layer{layer}'
    mean_score_df = pd.concat([mean_score_df, tmp_df])

# Save the results as a CSV
mean_score_df.to_csv(f'{data_dir}/examples_mean_{approach}_scores.csv', index=False)

del(tmp_df, tmp_dict, tmp_list, results, head, layer, f, i)

## Heatmap

In [ ]:
# Calculate mean score for each layer
tmp_mean_df = mean_score_df.groupby('layer').mean()

# Normalize the scores within each layer
tmp_mean_df = tmp_mean_df.apply(lambda x: (x - x.min()) / (x.max() - x.min()), axis=1)

# Sort layers
tmp_mean_df = tmp_mean_df.reindex(sorted(tmp_mean_df.index, key=lambda x: int(x[5:])))

# Plotting
plt.figure(1, figsize=(9, 6))
sns.set(color_codes=True)
sns.set(font_scale=0.9)
ax = sns.heatmap(tmp_mean_df, cmap='GnBu', cbar_kws={'label': 'Scale'})
ax.set(ylabel="Layers", xlabel="Heads")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, horizontalalignment='right')
ax.set_yticklabels(ax.get_yticklabels(), rotation=45)

plt.savefig(f"{data_dir}/scgpt_mean_attention_heatmap.png", dpi=300, bbox_inches='tight')

plt.show()

tmp_mean_df.to_csv(f'{data_dir}/tmp_mean_{approach}_scores.csv', index=False)

# Clean up
del(tmp_mean_df, ax)

 # Feature Matrix

In [ ]:
# Function to concatenate gene names or scores from a list
def concatenate_items(items):
    return ','.join(str(item) for item in items)

# Create a CSV file to hold all the data
output_csv_path = os.path.join(data_dir, 'compiled_cells.csv')
with open(output_csv_path, 'w', newline='') as csvfile:
    csvwriter = csv.writer(csvfile)
    # Write the header row
    headers = ['gene_sequence'] + ['label'] + ['expression'] + [f'layer{layer}_head{head}' for layer in range(num_layers) for head in range(num_heads)]
    csvwriter.writerow(headers)

    # For each cell, write a row in the CSV file
    for cell_index in range(num_cells):
        row = []
        for layer in range(num_layers):
            for head in range(num_heads):

                # attn_val = layers_data[0][0][0][0]
                # genes = layers_data[0][0][0][1]
                # label = layers_data[0][0][0][2]
                # exp_val = layers_data[0][0][0][3]
                
                # Label
                label = layers_data[layer][head][cell_index][2]
                
                # Expression Values
                expression_vals = layers_data[layer][head][cell_index][3]
                expression_seq = concatenate_items(expression_vals)

                # Gene Names
                gene_names = layers_data[layer][head][cell_index][1]
                gene_seq = concatenate_items(gene_names)

                # Attention Scores
                attn_scores = layers_data[layer][head][cell_index][0]
                attn_seq = concatenate_items(attn_scores)

                row.append(attn_seq)

        # print([gene_seq, row])
        csvwriter.writerow([gene_seq] + [label] + [expression_seq] + row)

In [ ]:
cells_path = os.path.join(data_dir, 'compiled_cells.csv')
df = pd.read_csv(cells_path)

df.head()

In [ ]:
# Get the first element (assuming comma-separated values)
gene_seq = df['gene_sequence'].iloc[0]

# # Split the string by comma, remove whitespace, and convert to float (handling errors)
num_genes = sum(1 for _ in [(x.strip()) for x in gene_seq.split(',')])
print(num_genes)

exp_seq = df['expression'].iloc[0]
num_exp = sum(1 for _ in [(x.strip()) for x in exp_seq.split(',')])
print(num_exp)

## Add Position as Feature

In [ ]:
cells_path = os.path.join(data_dir, 'compiled_cells.csv')
cells = pd.read_csv(cells_path)

cells.head()

In [ ]:
seq = df['gene_sequence'].iloc[0]
len_seq = sum(1 for _ in [(x.strip()) for x in seq.split(',')])

total_cells = len_seq

# Calculate the lengths of the thirds
third_length = total_cells // 3
last_third_start = 2 * third_length

# If total_cells is not perfectly divisible by 3, adjust the last third to include any extra cells
extra_cells = total_cells % 3
if extra_cells != 0:
    last_third_start += extra_cells - 1

# Generate sequences for first, middle, and last thirds
first_third = ["1"] * third_length + ["0"] * (total_cells - third_length)
middle_third = ["0"] * third_length + ["1"] * third_length + ["0"] * (total_cells - 2 * third_length)
last_third = ["0"] * last_third_start + ["1"] * (total_cells - last_third_start)

first_third_seq = ",".join(map(str, first_third))
middle_third_seq = ",".join(map(str, middle_third))
last_third_seq = ",".join(map(str, last_third))

cells['position_first_third'] = first_third_seq
cells['position_middle_third'] = middle_third_seq
cells['position_last_third'] = last_third_seq

In [ ]:
# Get the first element (assuming comma-separated values)
gene_seq = cells['position_first_third'].iloc[0]

# # Split the string by comma, remove whitespace, and convert to float (handling errors)
num_genes = sum(1 for _ in [(x.strip()) for x in gene_seq.split(',')])
print(num_genes)

exp_seq = cells['position_middle_third'].iloc[0]
num_exp = sum(1 for _ in [(x.strip()) for x in exp_seq.split(',')])
print(num_exp)

In [ ]:
# Calculate total number of rows in the dataframe
total_rows = len(cells)

# # Generate the sequence of numbers from 1 to 499
sequence = ",".join(map(str, range(1, 501)))

# Create a list where each element is the sequence, replicated for each row in the dataframe
position_sequence = [sequence for _ in range(total_rows)]

# # Assign this list of lists to the 'position' column in your dataframe
cells['position'] = position_sequence

cells.head()

In [ ]:
# Export compiled_cells.csv
cells.to_csv(cells_path, index=False, sep=';')

 # Concatenate Gene Sets

- H = hallmark
- C8 = cell type signature
- C5 = ontology

In [ ]:
# Import compiled_cells.csv
cells_path = os.path.join(data_dir, 'compiled_cells.csv')
cells = pd.read_csv(cells_path, sep=';')

cells.head()

In [ ]:
# List of geneset files
# Download from MSigDB: https://igordot.github.io/msigdbr/
genesets = ['H_geneset.csv', 'c8_geneset.csv', 'c5_geneset.csv']

## Generate features

In [ ]:
def generate_binary_string(gene_sequence, gene_set):
    # Convert gene_sequence into a list
    ordered_genes = gene_sequence.split(',')
    
    # Generate binary string
    binary_string = ['1' if gene in gene_set else '0' for gene in ordered_genes]
    
    # Convert binary_string list to a string
    binary_string = ','.join(binary_string)

    return(binary_string)

In [ ]:
# Iterate over genesets
for geneset in genesets:
    geneset_path = os.path.join(base, 'genesets', geneset)
    print('Geneset File Path:', geneset_path)

    # Read geneset file with specifying data types to avoid DtypeWarning
    geneset_raw = pd.read_csv(geneset_path, dtype={'column_name': str})

    # Extract unique gene set names
    gene_set_names = geneset_raw['gs_name'].unique()

    # Create a dictionary with gene set names as keys and their respective gene symbols as values
    gene_set_dict = {gs_name: set(geneset_raw.loc[geneset_raw['gs_name'] == gs_name, 'gene_symbol']) for gs_name in gene_set_names}

    # Apply the function for each gene set and append the results as new columns to cells DataFrame
    for gs_name, gene_set in gene_set_dict.items():
        cells[gs_name] = cells['gene_sequence'].apply(lambda x: generate_binary_string(x, gene_set))

    # Drop gene set file data from memory to conserve memory
    del geneset_raw

In [ ]:
# Export compiled_cells_features.csv
cells_features_path = os.path.join(data_dir, 'compiled_cells-features.csv')
cells.to_csv(cells_features_path, index=False, sep=';')

## Standardize Faetures

In [ ]:
# Open compiled_cells.csv
compiled_cells_path = os.path.join(data_dir, 'compiled_cells.csv')
compiled_cells = pd.read_csv(compiled_cells_path, sep=';')

compiled_cells.head()

In [ ]:
# Open compiled_cells-features.csv
cells_features_path = os.path.join(data_dir, 'compiled_cells-features.csv')
cells_features = pd.read_csv(cells_features_path, sep=';')

cells_features.head()

In [ ]:
# Get column names
column_names = cells_features.columns[103:].tolist()
print(f"Number of columns: {len(column_names)}")

# Print column names line by line
print("Column names:")
for col in column_names:
    print(col)

# Create a DataFrame with column names
feature_df = pd.DataFrame(column_names, columns=['feature'])

# Save the column names as a CSV file
output_path = os.path.join(data_dir, 'feature_list.csv')
feature_df.to_csv(output_path, index=False)

print(f"\nFeature list saved to: {output_path}")

In [ ]:
features_ms = [
    "BUSSLINGER_GASTRIC_IMMUNE_CELLS",
    "FAN_OVARY_CL8_MATURE_CUMULUS_GRANULOSA_CELL_2",
    "MANNO_MIDBRAIN_NEUROTYPES_HENDO",
    "MANNO_MIDBRAIN_NEUROTYPES_HGABA",
    "MANNO_MIDBRAIN_NEUROTYPES_HPERIC",
    "MURARO_PANCREAS_DUCTAL_CELL",
    "TRAVAGLINI_LUNG_PROLIFERATING_MACROPHAGE_CELL",
    "TRAVAGLINI_LUNG_PROXIMAL_CILIATED_CELL",
    "GOBP_ANATOMICAL_STRUCTURE_FORMATION_INVOLVED_IN_MORPHOGENESIS",
    "GOBP_BIOLOGICAL_ADHESION",
    "GOBP_CATION_TRANSPORT",
    "GOBP_CELLULAR_MACROMOLECULE_LOCALIZATION",
    "GOBP_CELLULAR_RESPONSE_TO_ENDOGENOUS_STIMULUS",
    "GOBP_CELLULAR_RESPONSE_TO_STRESS",
    "GOBP_CELL_CELL_SIGNALING",
    "GOBP_CELL_MIGRATION",
    "GOBP_CELL_PROJECTION_ORGANIZATION",
    "GOBP_CENTRAL_NERVOUS_SYSTEM_DEVELOPMENT",
    "GOBP_CHEMICAL_HOMEOSTASIS",
    "GOBP_CIRCULATORY_SYSTEM_DEVELOPMENT",
    "GOBP_CYTOSKELETON_ORGANIZATION",
    "GOBP_ESTABLISHMENT_OF_PROTEIN_LOCALIZATION",
    "GOBP_HOMEOSTATIC_PROCESS",
    "GOBP_INTRACELLULAR_TRANSPORT",
    "GOBP_ION_TRANSMEMBRANE_TRANSPORT",
    "GOBP_ION_TRANSPORT",
    "GOBP_LOCOMOTION",
    "GOBP_NEGATIVE_REGULATION_OF_RESPONSE_TO_STIMULUS",
    "GOBP_NEGATIVE_REGULATION_OF_SIGNALING",
    "GOBP_NERVOUS_SYSTEM_PROCESS",
    "GOBP_NEUROGENESIS",
    "GOBP_NEURON_DEVELOPMENT",
    "GOBP_NEURON_DIFFERENTIATION",
    "GOBP_NITROGEN_COMPOUND_TRANSPORT",
    "GOBP_ORGANONITROGEN_COMPOUND_BIOSYNTHETIC_PROCESS",
    "GOBP_PHOSPHORYLATION",
    "GOBP_POSITIVE_REGULATION_OF_CELLULAR_BIOSYNTHETIC_PROCESS",
    "GOBP_POSITIVE_REGULATION_OF_DEVELOPMENTAL_PROCESS",
    "GOBP_POSITIVE_REGULATION_OF_MOLECULAR_FUNCTION",
    "GOBP_POSITIVE_REGULATION_OF_MULTICELLULAR_ORGANISMAL_PROCESS",
    "GOBP_POSITIVE_REGULATION_OF_NUCLEOBASE_CONTAINING_COMPOUND_METABOLIC_PROCESS",
    "GOBP_POSITIVE_REGULATION_OF_PROTEIN_METABOLIC_PROCESS",
    "GOBP_POSITIVE_REGULATION_OF_RNA_METABOLIC_PROCESS",
    "GOBP_POSITIVE_REGULATION_OF_SIGNALING",
    "GOBP_PROGRAMMED_CELL_DEATH",
    "GOBP_PROTEIN_CONTAINING_COMPLEX_ORGANIZATION",
    "GOBP_PROTEOLYSIS",
    "GOBP_REGULATION_OF_CELL_DEATH",
    "GOBP_REGULATION_OF_CELL_DIFFERENTIATION",
    "GOBP_REGULATION_OF_CELL_POPULATION_PROLIFERATION",
    "GOBP_REGULATION_OF_INTRACELLULAR_SIGNAL_TRANSDUCTION",
    "GOBP_REGULATION_OF_MULTICELLULAR_ORGANISMAL_DEVELOPMENT",
    "GOBP_REGULATION_OF_PHOSPHORUS_METABOLIC_PROCESS",
    "GOBP_REGULATION_OF_PROTEIN_MODIFICATION_PROCESS",
    "GOBP_REGULATION_OF_TRANSPORT",
    "GOBP_REPRODUCTION",
    "GOBP_RESPONSE_TO_ABIOTIC_STIMULUS",
    "GOBP_RESPONSE_TO_ENDOGENOUS_STIMULUS",
    "GOBP_RESPONSE_TO_OXYGEN_CONTAINING_COMPOUND",
    "GOBP_SMALL_MOLECULE_METABOLIC_PROCESS",
    "GOBP_TISSUE_DEVELOPMENT",
    "GOBP_TRANSMEMBRANE_TRANSPORT",
    "GOBP_VESICLE_MEDIATED_TRANSPORT",
    "GOCC_ENDOPLASMIC_RETICULUM",
    "GOCC_ENVELOPE",
    "GOCC_GOLGI_APPARATUS",
    "GOCC_INTRINSIC_COMPONENT_OF_PLASMA_MEMBRANE",
    "GOCC_MEMBRANE_PROTEIN_COMPLEX",
    "GOCC_MITOCHONDRION",
    "GOCC_NEURON_PROJECTION",
    "GOCC_PLASMA_MEMBRANE_REGION",
    "GOCC_SECRETORY_VESICLE",
    "GOCC_SYNAPSE",
    "GOCC_VESICLE_MEMBRANE",
    "GOMF_IDENTICAL_PROTEIN_BINDING",
    "GOMF_MOLECULAR_FUNCTION_REGULATOR",
    "GOMF_PROTEIN_CONTAINING_COMPLEX_BINDING",
    "GOMF_RIBONUCLEOTIDE_BINDING",
    "GOMF_SIGNALING_RECEPTOR_BINDING",
    "GOMF_TRANSPORTER_ACTIVITY"
]

In [ ]:
features_pancreas = [
    "DESCARTES_FETAL_CEREBELLUM_VASCULAR_ENDOTHELIAL_CELLS",
    "GAO_LARGE_INTESTINE_ADULT_CJ_IMMUNE_CELLS",
    "HAY_BONE_MARROW_STROMAL",
    "MANNO_MIDBRAIN_NEUROTYPES_HENDO",
    "MANNO_MIDBRAIN_NEUROTYPES_HMGL",
    "MANNO_MIDBRAIN_NEUROTYPES_HPERIC",
    "MURARO_PANCREAS_ACINAR_CELL",
    "MURARO_PANCREAS_DUCTAL_CELL",
    "MURARO_PANCREAS_MESENCHYMAL_STROMAL_CELL",
    "TRAVAGLINI_LUNG_PROLIFERATING_MACROPHAGE_CELL",
    "GOBP_ANATOMICAL_STRUCTURE_FORMATION_INVOLVED_IN_MORPHOGENESIS",
    "GOBP_ANIMAL_ORGAN_MORPHOGENESIS",
    "GOBP_BIOLOGICAL_ADHESION",
    "GOBP_BIOLOGICAL_PROCESS_INVOLVED_IN_INTERSPECIES_INTERACTION_BETWEEN_ORGANISMS",
    "GOBP_CELLULAR_RESPONSE_TO_ENDOGENOUS_STIMULUS",
    "GOBP_CELLULAR_RESPONSE_TO_OXYGEN_CONTAINING_COMPOUND",
    "GOBP_CELLULAR_RESPONSE_TO_STRESS",
    "GOBP_CELL_ACTIVATION",
    "GOBP_CELL_CELL_ADHESION",
    "GOBP_CELL_CELL_SIGNALING",
    "GOBP_CELL_CYCLE",
    "GOBP_CELL_MIGRATION",
    "GOBP_CELL_PROJECTION_ORGANIZATION",
    "GOBP_CHEMICAL_HOMEOSTASIS",
    "GOBP_CIRCULATORY_SYSTEM_DEVELOPMENT",
    "GOBP_CYTOSKELETON_ORGANIZATION",
    "GOBP_DEFENSE_RESPONSE",
    "GOBP_ENZYME_LINKED_RECEPTOR_PROTEIN_SIGNALING_PATHWAY",
    "GOBP_EPITHELIUM_DEVELOPMENT",
    "GOBP_HOMEOSTATIC_PROCESS",
    "GOBP_IMMUNE_RESPONSE",
    "GOBP_INFLAMMATORY_RESPONSE",
    "GOBP_ION_TRANSPORT",
    "GOBP_LIPID_METABOLIC_PROCESS",
    "GOBP_LOCOMOTION",
    "GOBP_NEGATIVE_REGULATION_OF_BIOSYNTHETIC_PROCESS",
    "GOBP_NEGATIVE_REGULATION_OF_CELL_DEATH",
    "GOBP_NEGATIVE_REGULATION_OF_MOLECULAR_FUNCTION",
    "GOBP_NEGATIVE_REGULATION_OF_MULTICELLULAR_ORGANISMAL_PROCESS",
    "GOBP_NEGATIVE_REGULATION_OF_RESPONSE_TO_STIMULUS",
    "GOBP_NEGATIVE_REGULATION_OF_SIGNALING",
    "GOBP_NEUROGENESIS",
    "GOBP_NEURON_DIFFERENTIATION",
    "GOBP_NITROGEN_COMPOUND_TRANSPORT",
    "GOBP_PHOSPHORYLATION",
    "GOBP_POSITIVE_REGULATION_OF_CELLULAR_BIOSYNTHETIC_PROCESS",
    "GOBP_POSITIVE_REGULATION_OF_CELL_POPULATION_PROLIFERATION",
    "GOBP_POSITIVE_REGULATION_OF_DEVELOPMENTAL_PROCESS",
    "GOBP_POSITIVE_REGULATION_OF_GENE_EXPRESSION",
    "GOBP_POSITIVE_REGULATION_OF_MOLECULAR_FUNCTION",
    "GOBP_POSITIVE_REGULATION_OF_MULTICELLULAR_ORGANISMAL_PROCESS",
    "GOBP_POSITIVE_REGULATION_OF_NUCLEOBASE_CONTAINING_COMPOUND_METABOLIC_PROCESS",
    "GOBP_POSITIVE_REGULATION_OF_PROTEIN_METABOLIC_PROCESS",
    "GOBP_POSITIVE_REGULATION_OF_RNA_METABOLIC_PROCESS",
    "GOBP_POSITIVE_REGULATION_OF_SIGNALING",
    "GOBP_PROGRAMMED_CELL_DEATH",
    "GOBP_PROTEOLYSIS",
    "GOBP_REGULATION_OF_ANATOMICAL_STRUCTURE_MORPHOGENESIS",
    "GOBP_REGULATION_OF_CELLULAR_COMPONENT_MOVEMENT",
    "GOBP_REGULATION_OF_CELL_ADHESION",
    "GOBP_REGULATION_OF_CELL_DEATH",
    "GOBP_REGULATION_OF_CELL_DIFFERENTIATION",
    "GOBP_REGULATION_OF_CELL_POPULATION_PROLIFERATION",
    "GOBP_REGULATION_OF_IMMUNE_SYSTEM_PROCESS",
    "GOBP_REGULATION_OF_INTRACELLULAR_SIGNAL_TRANSDUCTION",
    "GOBP_REGULATION_OF_MULTICELLULAR_ORGANISMAL_DEVELOPMENT",
    "GOBP_REGULATION_OF_PHOSPHORUS_METABOLIC_PROCESS",
    "GOBP_REGULATION_OF_PROTEIN_MODIFICATION_PROCESS",
    "GOBP_REGULATION_OF_PROTEIN_PHOSPHORYLATION",
    "GOBP_REGULATION_OF_RESPONSE_TO_EXTERNAL_STIMULUS",
    "GOBP_REGULATION_OF_RESPONSE_TO_STRESS",
    "GOBP_REGULATION_OF_TRANSPORT",
    "GOBP_REPRODUCTION",
    "GOBP_RESPONSE_TO_ABIOTIC_STIMULUS",
    "GOBP_RESPONSE_TO_CYTOKINE",
    "GOBP_RESPONSE_TO_ENDOGENOUS_STIMULUS",
    "GOBP_RESPONSE_TO_LIPID",
    "GOBP_RESPONSE_TO_OXYGEN_CONTAINING_COMPOUND",
    "GOBP_SMALL_MOLECULE_METABOLIC_PROCESS",
    "GOBP_TISSUE_DEVELOPMENT",
    "GOBP_TRANSMEMBRANE_TRANSPORT",
    "GOBP_TUBE_DEVELOPMENT",
    "GOBP_TUBE_MORPHOGENESIS",
    "GOBP_VASCULATURE_DEVELOPMENT",
    "GOBP_VESICLE_MEDIATED_TRANSPORT",
    "GOCC_CELL_SURFACE",
    "GOCC_ENDOPLASMIC_RETICULUM",
    "GOCC_GOLGI_APPARATUS",
    "GOCC_INTRINSIC_COMPONENT_OF_PLASMA_MEMBRANE",
    "GOCC_PLASMA_MEMBRANE_REGION",
    "GOMF_IDENTICAL_PROTEIN_BINDING",
    "GOMF_MOLECULAR_FUNCTION_REGULATOR",
    "GOMF_MOLECULAR_TRANSDUCER_ACTIVITY",
    "GOMF_PROTEIN_CONTAINING_COMPLEX_BINDING",
    "GOMF_SEQUENCE_SPECIFIC_DNA_BINDING",
    "GOMF_SIGNALING_RECEPTOR_BINDING",
    "GOMF_TRANSCRIPTION_REGULATOR_ACTIVITY",
    "HP_ABNORMAL_RESPIRATORY_SYSTEM_MORPHOLOGY"
]

In [ ]:
if dataset == 'ms':
    columns_to_consider = ['gene_sequence'] + features_ms
else:
    columns_to_consider = ['gene_sequence'] + features_pancreas
    
print('Number of Columns:', len(columns_to_consider))

In [ ]:
# Filtering the dataframe to only include these columns
filtered_cells = cells_features[columns_to_consider]

print('Shape:', filtered_cells.shape)
filtered_cells.head()

In [ ]:
# Export compiled_cells_all-features.csv
num_col = filtered_cells.shape[1]
filtered_cells_path = os.path.join(data_dir, f'filt_features_{num_col}.csv')
filtered_cells.to_csv(filtered_cells_path, index=False, sep=';')

In [ ]:
# Extract the column names directly
old_col = list(filtered_cells.columns)
# Create a new DataFrame with these column names as row values in a single column
rename_df = pd.DataFrame(old_col, columns=['old_col'])
# Display the first few rows of the new DataFrame
rename_df.head()

In [ ]:
# Open new column names
col_df_path = os.path.join(data_dir, f'column_names_final.csv') # rename existing columns for readability, col: old_col, new_col
col_df = pd.read_csv(col_df_path)
col_df.head(10)

In [ ]:
filtered_cells.rename(columns=dict(zip(col_df['old_col'], col_df['new_col'])), inplace=True)
filtered_cells.head()

In [ ]:
# Merge the DataFrames on the 'gene_sequence' column
num_col = len(columns_to_consider)
print('Number of Features:', num_col)
merge_filtered_cells = pd.merge(compiled_cells, filtered_cells, on='gene_sequence', how='inner')
merge_filtered_cells.head()

In [ ]:
final_df_path = os.path.join(data_dir, f'scgpt_{model}_{dataset}-features_{num_col}.csv')
print(final_df_path)
merge_filtered_cells.to_csv(final_df_path, index=False, sep=';')